## dog breed classification real project

In [1]:
import tensorflow as tf
import tensorflow_hub as hub
import tf_keras
from fastapi import FastAPI, UploadFile, File
from PIL import Image
import numpy as np
import io
from fastapi.responses import HTMLResponse

#create the app
app = FastAPI()

model = model = tf_keras.models.load_model(
    '/content/drive/MyDrive/dog_breed/models/20263609/06/26-143638-1000_images_effectivenetv2b0.h5',
    custom_objects={'KerasLayer': hub.KerasLayer})
print("model_downloaded")

model_downloaded


In [2]:
def prepare_image(image_bytes):
    # Convert raw bytes to a TensorFlow string tensor
    img_tensor = tf.constant(image_bytes)

    # Decode the JPEG image into a numerical tensor with 3 color channels
    image = tf.image.decode_jpeg(img_tensor, channels=3)

    # Convert the color channel values from 0-255 to 0-1
    image = tf.image.convert_image_dtype(image, tf.float32)

    # Resize our image to the desired image size
    image = tf.image.resize(image, size=[224, 224])

    # Add batch dimension: shape becomes (1, 224, 224, 3)
    image = tf.expand_dims(image, axis=0)

    return image

In [3]:
# Replace these names with the actual dog breeds your model was trained on
# Make sure the order exactly matches your training data labels
class_names = ['affenpinscher', 'afghan_hound', 'african_hunting_dog', 'airedale',
       'american_staffordshire_terrier', 'appenzeller',
       'australian_terrier', 'basenji', 'basset', 'beagle',
       'bedlington_terrier', 'bernese_mountain_dog',
       'black-and-tan_coonhound', 'blenheim_spaniel', 'bloodhound',
       'bluetick', 'border_collie', 'border_terrier', 'borzoi',
       'boston_bull', 'bouvier_des_flandres', 'boxer',
       'brabancon_griffon', 'briard', 'brittany_spaniel', 'bull_mastiff',
       'cairn', 'cardigan', 'chesapeake_bay_retriever', 'chihuahua',
       'chow', 'clumber', 'cocker_spaniel', 'collie',
       'curly-coated_retriever', 'dandie_dinmont', 'dhole', 'dingo',
       'doberman', 'english_foxhound', 'english_setter',
       'english_springer', 'entlebucher', 'eskimo_dog',
       'flat-coated_retriever', 'french_bulldog', 'german_shepherd',
       'german_short-haired_pointer', 'giant_schnauzer',
       'golden_retriever', 'gordon_setter', 'great_dane',
       'great_pyrenees', 'greater_swiss_mountain_dog', 'groenendael',
       'ibizan_hound', 'irish_setter', 'irish_terrier',
       'irish_water_spaniel', 'irish_wolfhound', 'italian_greyhound',
       'japanese_spaniel', 'keeshond', 'kelpie', 'kerry_blue_terrier',
       'komondor', 'kuvasz', 'labrador_retriever', 'lakeland_terrier',
       'leonberg', 'lhasa', 'malamute', 'malinois', 'maltese_dog',
       'mexican_hairless', 'miniature_pinscher', 'miniature_poodle',
       'miniature_schnauzer', 'newfoundland', 'norfolk_terrier',
       'norwegian_elkhound', 'norwich_terrier', 'old_english_sheepdog',
       'otterhound', 'papillon', 'pekinese', 'pembroke', 'pomeranian',
       'pug', 'redbone', 'rhodesian_ridgeback', 'rottweiler',
       'saint_bernard', 'saluki', 'samoyed', 'schipperke',
       'scotch_terrier', 'scottish_deerhound', 'sealyham_terrier',
       'shetland_sheepdog', 'shih-tzu', 'siberian_husky', 'silky_terrier',
       'soft-coated_wheaten_terrier', 'staffordshire_bullterrier',
       'standard_poodle', 'standard_schnauzer', 'sussex_spaniel',
       'tibetan_mastiff', 'tibetan_terrier', 'toy_poodle', 'toy_terrier',
       'vizsla', 'walker_hound', 'weimaraner', 'welsh_springer_spaniel',
       'west_highland_white_terrier', 'whippet',
       'wire-haired_fox_terrier', 'yorkshire_terrier']

@app.post("/predict")
async def predict_dog_breed(file: UploadFile = File(...)):
    # Read the file uploaded by the user
    image_bytes = await file.read()

    # Process the image to match model input shape
    processed_image = prepare_image(image_bytes)

    # Predict using the loaded model
    predictions = model.predict(processed_image)

    # Get the index of the highest probability
    predicted_class_index = np.argmax(predictions[0])

    # Get the actual highest probability value (confidence)
    confidence = np.max(predictions[0])

    # Map the index to the actual breed name
    predicted_breed = class_names[predicted_class_index]

    # Make percentage
    confidence_percentage = round(float(confidence) * 100, 2)

    # Return a JSON response
    return {
        "filename": file.filename,
        "breed": predicted_breed, # تم تعديل الاسم هنا ليطابق الواجهة
        "confidence": confidence_percentage
    }

In [4]:
len(class_names)

120

In [5]:
import nest_asyncio
import uvicorn
import threading
from google.colab import output

# Allow asyncio to run within Jupyter/Colab environments
nest_asyncio.apply()

def run_server():
    # Run the FastAPI app on port 8000
    uvicorn.run(app, host="0.0.0.0", port=8000)

# Run the server in a background thread so the notebook doesn't freeze
server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()

# Create a public link to access the server running on Colab
print("Server is running!")
print("Click the link below to open the API:")
output.serve_kernel_port_as_window(8000)

Server is running!
Click the link below to open the API:
Try `serve_kernel_port_as_iframe` instead. 


<IPython.core.display.Javascript object>

In [6]:
@app.get("/", response_class=HTMLResponse)
async def get_webpage():
    # Read and return the HTML file
    with open("/content/drive/MyDrive/index.html", "r", encoding="utf-8") as f:
        return f.read()

In [7]:
import urllib

# Get the IP address to use as a password for localtunnel
ip_password = urllib.request.urlopen('https://ipv4.icanhazip.com').read().decode('utf8').strip("\n")
print("Your Password is:", ip_password)

# Install localtunnel
!npm install -g localtunnel

INFO:     Started server process [62459]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


Your Password is: 34.87.28.240
⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦
changed 22 packages in 5s
⠦
⠦3 packages are looking for funding
⠦  run `npm fund` for details
⠦

In [ ]:
import nest_asyncio
import uvicorn
import threading

# Allow asyncio in Colab
nest_asyncio.apply()

# Run FastAPI in the background
def run_server():
    uvicorn.run(app, host="0.0.0.0", port=8000)

threading.Thread(target=run_server, daemon=True).start()

# Create a public link using localtunnel
!npx localtunnel --port 8000

INFO:     Started server process [62459]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
ERROR:    [Errno 98] error while attempting to bind on address ('0.0.0.0', 8000): address already in use
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.


⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴your url is: https://cold-poets-join.loca.lt
INFO:     83.171.207.215:0 - "GET / HTTP/1.1" 200 OK
INFO:     83.171.207.215:0 - "GET / HTTP/1.1" 200 OK
INFO:     83.171.207.215:0 - "GET /favicon.ico HTTP/1.1" 404 Not Found
1/1 [==============================] - 1s 731ms/step
INFO:     83.171.207.215:0 - "POST /predict HTTP/1.1" 200 OK
1/1 [==============================] - 0s 72ms/step
INFO:     83.171.207.215:0 - "POST /predict HTTP/1.1" 200 OK
